# 21 · RAG-Fusion：把 Multi-Query 组装成产品

> RAG-Fusion = 原问题 + LLM 生成多个变体 → 多路并行检索 → **RRF 融合** → Top-K 给 LLM。是 Multi-Query(19) 与 Hybrid 融合(17) 的组合拳。

**本文件覆盖知识点**：RAG-Fusion 全流程 / Multi-query / Parallel Retrieval / RRF / Weighted RRF / Query Weight / Fusion Strategy

```text
Original Query
      │ LLM
      ▼
Generate Multiple Queries
      │
      ▼
Multiple Retrieval (并行)
      │
      ▼
RRF 融合
      │
      ▼
Top-K → LLM → Answer
```

In [ ]:
# ===== 本课共用：真实检索底座 =====
# 真语料(data/) → 真切分 → 真向量(text-embedding-v3) → 真索引(FAISS + BM25)
# → 真重排(qwen3-rerank) → 真生成(qwen-plus)。各课在这个底座上演示自己的知识点。
#
# 说明：向量按内容哈希缓存在 .cache/emb.npz（首次真调、之后复用，避免反复花 token）。
# 没配 DASHSCOPE_API_KEY 时仍可用：向量直接从缓存读（是此前真实调用的结果），
# 但需要现场调用模型的重排/生成会打印录制结果并提示配置方式。
from dotenv import load_dotenv; load_dotenv()
import os, re, json, time, hashlib
from pathlib import Path
import numpy as np

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY
_DATA = Path('data') if Path('data').is_dir() else Path.cwd() / 'data'
_CACHE_FILE = Path('.cache') / 'emb.npz'
EMBED_MODEL = 'text-embedding-v3'
RERANK_MODEL = 'qwen3-rerank'
NO_KEY_TIP = ('未配置 DASHSCOPE_API_KEY：需要现场调用模型的部分将展示此前真实调用的录制结果，'
              '在项目根 .env 配置后自动变为实时调用。')

def recorded(text, note=''):
    """无 Key 时展示「此前真实运行的录制结果」。内容来自真实调用，不是编造的假数据。"""
    print(NO_KEY_TIP)
    print('—— 录制结果%s ——' % ('（' + note + '）' if note else ''))
    print(text)

if not _HAS_KEY:
    print(NO_KEY_TIP)

# ---------- 1) 语料：读 data/ 全部 Markdown，按小节切块 ----------
# 注意：评测集.md 是「人工标注的答案」，不能进索引 —— 否则第 33 课评测时，
# 标注本身会被检索命中，指标虚高（数据泄漏）。这里显式排除。
_EXCLUDE = {'评测集.md'}

def load_chunks(chunk_size=300, overlap=60):
    """按「## 小节」切分，小节过长再按句子窗口滑切。返回 [{'i','text','source','section'}]"""
    out = []
    for p in sorted(_DATA.glob('*.md')):
        if p.name in _EXCLUDE:
            continue
        section, buf = p.stem, []
        for line in p.read_text(encoding='utf-8').splitlines():
            if line.startswith('## '):
                if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
                section, buf = line[3:].strip(), [line]
            elif line.startswith('# '):
                section = line[2:].strip()
            else:
                buf.append(line)
        if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
    for i, c in enumerate(out):
        c['i'] = i
    return out

def _split_section(lines, section, source, chunk_size, overlap):
    """小节内容按句号聚合成 ~chunk_size 字的片段，相邻片段留 overlap 字重叠"""
    text = '\n'.join(lines).strip()
    if not text: return []
    sents = [s for s in re.split(r'(?<=[。！？\n])', text) if s.strip()]
    chunks, buf = [], ''
    for s in sents:
        if len(buf) + len(s) > chunk_size and buf:
            chunks.append(buf.strip())
            buf = buf[-overlap:] + s          # 保留尾部 overlap 字做上下文重叠
        else:
            buf += s
    if buf.strip(): chunks.append(buf.strip())
    return [{'text': c, 'source': source, 'section': section} for c in chunks]

# ---------- 2) 向量：真调 text-embedding-v3（分批 + 重试 + 内容哈希缓存）----------
def _load_cache():
    if not _CACHE_FILE.exists():
        return {}
    try:
        z = np.load(_CACHE_FILE, allow_pickle=False)
        return dict(zip(z['hashes'].tolist(), z['vectors']))
    except Exception as e:                      # 文件损坏（例如多进程同时写）：当空缓存重建，别让 notebook 挂掉
        print('向量缓存不可读(%s: %s)，将重新向量化：%s' % (type(e).__name__, e, _CACHE_FILE))
        return {}

def _save_cache(cache):
    """写盘前先与磁盘上已有内容合并，再原子替换 —— 避免多个进程同时跑时互相覆盖 / 写坏文件"""
    _CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    for k, v in _load_cache().items():
        cache.setdefault(k, v)
    hs = np.array(list(cache.keys()))
    vs = np.array([cache[h] for h in cache.keys()], dtype='float32')
    # 进程号唯一，别抢同一个临时文件；注意 np.savez_compressed 会自动补 .npz 后缀，临时名必须也是 .npz 结尾
    tmp = _CACHE_FILE.with_name('%s.%d.tmp.npz' % (_CACHE_FILE.stem, os.getpid()))
    np.savez_compressed(tmp, hashes=hs, vectors=vs)
    try:
        os.replace(tmp, _CACHE_FILE)            # 原子替换：别的进程读到的永远是完整文件
    except OSError:                             # 目标被占用时稍等再试
        time.sleep(0.2); os.replace(tmp, _CACHE_FILE)

def _key(text, model):
    return hashlib.sha1((model + '\x00' + text).encode('utf-8')).hexdigest()[:16]

def embed(texts, model=EMBED_MODEL, batch=10):
    """真调 Embedding；命中缓存则直接用（缓存来自真实调用）。返回已 L2 归一化的向量"""
    if isinstance(texts, str): texts = [texts]
    cache, todo = _load_cache(), []
    for t in texts:
        k = _key(t, model)
        if k not in cache and k not in [x[0] for x in todo]:
            todo.append((k, t))
    if todo and not _HAS_KEY:
        raise RuntimeError('本地缓存缺少 %d 条向量，且未配置 DASHSCOPE_API_KEY，无法现场向量化。'
                           '请在项目根 .env 配置 Key 后重跑。' % len(todo))
    if todo:
        from dashscope import TextEmbedding
        pending = todo
        while pending:                                  # 批次过大就减半重试
            b = pending[:batch]
            r = TextEmbedding.call(model=model, input=[t for _, t in b], api_key=_KEY)
            if r.status_code == 200:
                for (k, _), e in zip(b, sorted(r.output['embeddings'], key=lambda e: e['text_index'])):
                    cache[k] = np.array(e['embedding'], dtype='float32')
                pending = pending[len(b):]
            elif batch > 1:
                batch //= 2
            else:
                raise RuntimeError('向量化失败: %s %s' % (r.code, r.message))
        _save_cache(cache)
    v = np.array([cache[_key(t, model)] for t in texts], dtype='float32')
    return v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-10)

# ---------- 3) 索引：FAISS（归一化后内积=余弦）+ BM25 ----------
import faiss
from rank_bm25 import BM25Okapi

def tokenize(text):
    """中文用「单字 + 相邻双字」切词，无需外部分词器（与第 16 课一致）"""
    t = re.sub(r'\s+', '', text)
    return [t[i] for i in range(len(t))] + [t[i:i + 2] for i in range(len(t) - 1)]

CHUNKS = load_chunks()
VECS = embed([c['text'] for c in CHUNKS])
INDEX = faiss.IndexFlatIP(VECS.shape[1]); INDEX.add(VECS)
BM25 = BM25Okapi([tokenize(c['text']) for c in CHUNKS])
print('语料就绪：%d 篇文档 → %d 个片段，向量维度 %d' % (len({c['source'] for c in CHUNKS}), len(CHUNKS), VECS.shape[1]))

# ---------- 4) 检索：稠密 / 稀疏 / 混合（RRF 融合）----------
def dense_retrieve(query, k=5):
    sims, ids = INDEX.search(embed(query), k)
    return [dict(CHUNKS[i], score=float(s), from_='dense') for i, s in zip(ids[0], sims[0]) if i != -1]

def sparse_retrieve(query, k=5):
    scores = BM25.get_scores(tokenize(query))
    top = np.argsort(-scores)[:k]
    return [dict(CHUNKS[i], score=float(scores[i]), from_='bm25') for i in top if scores[i] > 0]

def hybrid_retrieve(query, k=5, rrf_k=60, pool=10):
    """RRF 融合：score = Σ 1/(rrf_k + rank)，只用名次不用原始分数，天然可比"""
    fused = {}
    for name, hits in (('dense', dense_retrieve(query, pool)), ('bm25', sparse_retrieve(query, pool))):
        for rank, h in enumerate(hits, 1):
            cur = fused.setdefault(h['i'], dict(h, score=0.0, from_=set()))
            cur['score'] += 1.0 / (rrf_k + rank)
            cur['from_'].add(name)
    return sorted(fused.values(), key=lambda x: -x['score'])[:k]

# ---------- 5) 重排：真调 DashScope TextReRank ----------
def rerank(query, docs, top_n=3, model=RERANK_MODEL):
    """docs 可以是字符串列表或检索结果 dict 列表；返回 [(文档, 相关性分数)]"""
    texts = [d['text'] if isinstance(d, dict) else d for d in docs]
    if not texts: return []
    if not _HAS_KEY:
        print(NO_KEY_TIP); return [(t, None) for t in texts[:top_n]]
    from dashscope import TextReRank
    r = TextReRank.call(model=model, query=query, documents=texts,
                        top_n=min(top_n, len(texts)), return_documents=False, api_key=_KEY)
    if r.status_code != 200:
        raise RuntimeError('重排失败: %s %s' % (r.code, r.message))
    return [(texts[it['index']], float(it['relevance_score'])) for it in r.output['results']]

# ---------- 6) 生成：qwen-plus（带重试）+ 结构化 JSON 输出 ----------
def chat(prompt, system='你是严谨的 RAG 助手：只依据给定资料回答，资料里没有的就直说不知道。',
         temperature=0.3, model='qwen-plus', retries=3):
    if not _HAS_KEY:
        return None
    from dashscope import Generation
    for attempt in range(retries):
        r = Generation.call(model=model, messages=[{'role': 'system', 'content': system},
                                                   {'role': 'user', 'content': prompt}],
                            temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            return r.output.choices[0].message.content
        if attempt == retries - 1:
            raise RuntimeError('生成失败: %s %s' % (r.code, r.message))
        time.sleep(1.5 * (attempt + 1))          # 限流类错误退避重试
    return None

def chat_json(prompt, system='只输出 JSON，不要任何解释或代码块标记。', retries=2, **kw):
    """要求模型输出 JSON 并解析；解析失败时把报错回喂再试一次"""
    for attempt in range(retries + 1):
        out = chat(prompt, system=system, **kw)
        if out is None: return None
        seg = out[out.find('{'): out.rfind('}') + 1]     # 容忍 ```json 包裹与前后废话
        try:
            return json.loads(seg)
        except Exception as e:
            if attempt == retries: raise
            prompt = prompt + '\n\n上次输出无法解析(%s)，请只输出合法 JSON。' % e
    return None


In [ ]:
# 真实 RAG-Fusion 全流程：LLM 改写查询 → 多路真实检索 → RRF 融合 → Top-K
# 与 19 课 Multi-Query 的区别：Fusion 不做结果并集，而是把多张榜单按「名次」用 RRF 合成一张榜，
# 所以各路检索器分数不必归一化、天然可比。
from collections import defaultdict

QUERY = '机器人答得不准、检索不到答案，怎么排查和优化？'
_RRF_K = 60

def generate_queries(q, n=3):
    """真调 qwen-plus：把原问题改写成 n 个侧重不同的检索查询（换说法 / 换角度，扩大召回面）"""
    obj = chat_json(
        '把下面的问题改写成 %d 个侧重不同的检索查询，用于在企业知识库里召回资料。\n'
        '要求：每个查询是一个完整短句、保留关键实体，分别从不同角度切入且互不重复。\n'
        '原始问题：%s\n'
        '只输出 JSON：{"queries": ["查询1", "查询2", "查询3"]}' % (n, q))
    qs = [s.strip() for s in (obj or {}).get('queries', []) if isinstance(s, str) and s.strip()]
    return qs[:n]

def rrf_fuse(rankings, weights=None, k=_RRF_K):
    """RRF：score(d) = Σ w_q / (k + rank_q(d))。只用名次不用原始分，多路分数天然可比。"""
    weights = weights or [1.0] * len(rankings)
    score = defaultdict(float)
    for w, rk in zip(weights, rankings):
        for rank, doc in enumerate(rk, 1):
            score[doc] += w / (k + rank)
    return sorted(score.items(), key=lambda kv: -kv[1])

# 无 Key 时用上次真实调用生成的改写查询（它们的向量已随首次真实检索写入缓存，检索仍走真实底座）
_RECORDED_QUERIES = [
    '机器人问答准确率低的常见原因及对应排查步骤有哪些？',
    '如何优化机器人检索模块以提升答案召回率和匹配精度？',
    '针对机器人答不准和检索失败问题，有哪些典型故障场景与解决方案？',
]

print('原问题：%s' % QUERY)
if _HAS_KEY:
    REWRITES = generate_queries(QUERY, n=3)
    print('\n① qwen-plus 真调生成的改写查询:')
else:
    REWRITES = _RECORDED_QUERIES
    recorded('\n'.join('   %s' % q for q in REWRITES),
             '录制于 2026-09-12，模型 qwen-plus（改写查询的向量已随录制时的真实检索写入缓存，下面检索仍走真实底座）')
    print('\n① 改写查询（录制结果）:')
QUERIES = [QUERY] + REWRITES
for i, q in enumerate(QUERIES):
    print('   Q%d = %s' % (i, q))

# ② 每路查询各自真实混合检索（底座 hybrid_retrieve = FAISS 向量 + BM25 的 RRF）
# 注：生产上这几路是并行的（n 倍延迟靠并行压回 1 倍）；这里串行跑，是为了让共享的
# 向量缓存保持单写者，避免多进程/多线程同时写坏 .cache/emb.npz。
print('\n② 各路真实混合检索 Top-5:')
rankings = {}
for q in QUERIES:
    hits = hybrid_retrieve(q, k=5)
    rankings[q] = [h['i'] for h in hits]
    print('   「%s」' % q)
    for r, h in enumerate(hits, 1):
        print('      %d. %s · %s :: %s' % (r, h['source'], h['section'], h['text'][:30].replace('\n', ' ')))

# ③ RRF 融合
TOP_K = 5
rankings_list = [rankings[q] for q in QUERIES]
fused = rrf_fuse(rankings_list)
print('\n③ RRF 融合后 Top-%d（被越多路命中，名次越靠前）:' % TOP_K)
for rank, (i, s) in enumerate(fused[:TOP_K], 1):
    c = CHUNKS[i]
    cov = [n for n, q in enumerate(QUERIES) if i in rankings[q]]
    print('   %d. RRF=%.5f  [%s · %s]  被 %s 命中' % (rank, s, c['source'], c['section'],
                                                     '/'.join('Q%d' % n for n in cov)))
    print('        %s' % c['text'][:56].replace('\n', ' '))

# ④ 融合前后排名对比
def _tag(i):
    return '%s§%s' % (CHUNKS[i]['source'].replace('.md', ''), CHUNKS[i]['section'])

base_ids = rankings[QUERY][:TOP_K]
fused_ids = [i for i, _ in fused[:TOP_K]]
print('\n④ 融合前后对比:')
print('   仅原问题 Q0 的 Top-5 : ' + ' | '.join(_tag(i) for i in base_ids))
print('   多路 RRF 融合后 Top-5: ' + ' | '.join(_tag(i) for i in fused_ids))
print('   融合新带进来的片段  : %d 个（改写查询从别的说法/角度捞回来的）'
      % len([i for i in fused_ids if i not in base_ids]))
union = sorted(set().union(*rankings_list))
print('   四路 Top-5 并集     : %d 个片段（只用原问题只有 %d 个）→ 改写把候选面撑大了'
      % (len(union), len(rankings[QUERY])))
print('   并集里单路漏掉、只能靠改写捞回的片段:')
for i in union:
    if i not in rankings[QUERY]:
        print('      [%s · %s] %s' % (CHUNKS[i]['source'], CHUNKS[i]['section'],
                                      CHUNKS[i]['text'][:44].replace('\n', ' ')))

# ⑤ Weighted RRF：原问题通常最贴题，给它更高权重（教学点 Query Weight）
weighted = rrf_fuse(rankings_list, weights=[2.0] + [1.0] * len(REWRITES))
print('\n⑤ Weighted RRF（原问题权重 2.0，改写各 1.0）Top-%d:' % TOP_K)
for rank, (i, s) in enumerate(weighted[:TOP_K], 1):
    print('   %d. RRFw=%.5f  [%s · %s]' % (rank, s, CHUNKS[i]['source'], CHUNKS[i]['section']))
print('   → 与等权榜相比顺序%s变化；这就是「原句更重要」时该调的旋钮。'
      % ('有' if [i for i, _ in weighted[:TOP_K]] != fused_ids else '无'))
print('\n→ Top-%d 就是最终交给 LLM 的上下文；RAG-Fusion 的收益来自改写扩大召回，'
      '代价是 n 倍检索开销，n 一般取 3~5。' % TOP_K)


## 融合策略再升级

| 策略 | 公式/做法 | 何时用 |
|------|-----------|--------|
| RRF | `Σ 1/(k+rank)` | 默认首选 |
| **Weighted RRF** | 给各查询榜单乘权重 `Σ w_q/(k+rank)` | 原问题比改写更重要 |
| Query Weight | 显式调原查询/改写查询的权重 | 有时原句最重要 |
| Score Fusion | 归一化后加权相加 | 想保留“分数强度” |

## 成本与质量

- **收益**：多角度召回，Recall 显著提高；
- **成本**：n 倍查询延迟/费用 → 用**并行**压缩延迟，n 一般 3~5；
- **验收**：仍要回到第 33/34 课评测看 Recall 提升是否值回成本。

## 小结

- RAG-Fusion = **多查询 + 并行检索 + RRF 融合**；
- 融合策略：RRF 起步，Weighted 优化，QoS 看评测；
- 这是“Query Transformation”向“精排前召回优化”的集大成。